# 1. Giới thiệu và mô tả các tính năng của Neo4j Graph Data Science (GDS)


**Neo4j Graph Data Science (GDS)** là một thư viện chuyên biệt của hệ quản trị cơ sở dữ liệu đồ thị Neo4j, được thiết kế nhằm hỗ trợ xây dựng, phân tích và khai thác tri thức từ dữ liệu dạng đồ thị ở quy mô lớn.  
GDS cung cấp tập hợp phong phú các thuật toán tối ưu cho phân tích đồ thị — bao gồm:

- **Centrality** (PageRank, Betweenness, Degree, …)
- **Community Detection** (Louvain, Label Propagation, Connected Components, …)
- **Similarity** (Node similarity, Jaccard, cosine)
- **Pathfinding** (Dijkstra, A*, Yen's K-Shortest Paths, …)
- **Machine Learning trên đồ thị** (node classification, link prediction,…)
- **Graph embedding** (FastRP, Node2Vec,…)

Thư viện tận dụng mô hình **in-memory graph projection**, cho phép xử lý nhanh hơn đáng kể so với truy vấn trực tiếp trên Neo4j. GDS hỗ trợ mạnh mẽ cho phân tích dữ liệu đồ thị, gợi ý sản phẩm, phát hiện gian lận, mô hình hoá mạng lưới và tối ưu hệ thống.

# 2. Hướng dẫn cài đặt Neo4j GDS

In [ ]:
## Cài đặt Python client (dùng trong Jupyter)
%pip install neo4j
%pip install graphdatascience

## Cài đặt Neo4j Desktop + Plugin GDS

1. Tải và cài đặt **Neo4j Desktop**  
2. Tạo một database mới  
3. Vào mục **Plugins** → Cài đặt **Graph Data Science**

# 3. Khởi chạy thuật toán tìm đường đi ngắn nhất

In [6]:
from graphdatascience import GraphDataScience
import pandas as pd

# cấu hình kết nối — chỉnh lại URI / user / pass theo máy bạn
gds = GraphDataScience("bolt://127.0.0.1:7687", auth=("neo4j", "123456aA@"))

def shortest_path_dijkstra_by_names(start_name: str, end_name: str, weight_prop: str = "weight"):
    # lấy id của 2 nút theo property name (label Person)
    q = "MATCH (s:Person {name:$s}), (t:Person {name:$t}) RETURN id(s) AS sid, id(t) AS tid LIMIT 1"
    ids = gds.run_cypher(q, {"s": start_name, "t": end_name})
    if ids.empty:
        raise ValueError(f"Không tìm thấy một trong hai node: {start_name}, {end_name}")
    sid = int(ids.loc[0, "sid"])
    tid = int(ids.loc[0, "tid"])

    # đảm bảo projection tạm 'tmpGraph' không tồn tại
    exists = gds.run_cypher("CALL gds.graph.exists('tmpGraph') YIELD exists")
    if not exists.empty and exists.loc[0, "exists"]:
        gds.run_cypher("CALL gds.graph.drop('tmpGraph') YIELD graphName")

    # tạo cypher projection (project tất cả node/quan hệ, đặt weight mặc định = 1 nếu không có)
    gds.run_cypher(f"""
CALL gds.graph.project.cypher(
  'tmpGraph',
  'MATCH (n) RETURN id(n) AS id',
  'MATCH (a)-[r]->(b) RETURN id(a) AS source, id(b) AS target, coalesce(r.{weight_prop}, 1) AS weight'
)
""")

    # gọi Dijkstra (stream). Một số phiên bản GDS trả các trường khác nhau; trường map dưới đây là chuẩn phổ biến.
    df = gds.run_cypher("""
CALL gds.shortestPath.dijkstra.stream('tmpGraph', {sourceNode: $s, targetNode: $t, relationshipWeightProperty: 'weight'})
YIELD nodeIds, costs, totalCost
RETURN nodeIds, costs, totalCost
""", {"s": sid, "t": tid})

    if df.empty:
        # không có đường đi
        gds.run_cypher("CALL gds.graph.drop('tmpGraph') YIELD graphName")
        return None

    row = df.loc[0]
    node_ids = row["nodeIds"]
    costs = row["costs"]
    total = row["totalCost"]

    # chuyển id -> tên để dễ đọc
    names_df = gds.run_cypher("UNWIND $ids AS id RETURN id, gds.util.asNode(id).name AS name", {"ids": node_ids})
    id_to_name = dict(zip(names_df["id"], names_df["name"]))
    path_names = [id_to_name[i] for i in node_ids]

    # dọn projection tạm
    gds.run_cypher("CALL gds.graph.drop('tmpGraph') YIELD graphName")

    return {"node_ids": node_ids, "path": path_names, "costs": costs, "totalCost": total}

# VD sử dụng: sửa tên theo graph của bạn
res = shortest_path_dijkstra_by_names("Angelina Jolie", "Brad Pitt", weight_prop="weight")
if res is None:
    print("Không tìm thấy đường đi.")
else:
    print("Path (names):", res["path"])
    print("Node ids:", res["node_ids"])
    print("Costs per hop:", res["costs"])
    print("Total cost:", res["totalCost"])

Path (names): ['Angelina Jolie', 'Brad Pitt']
Node ids: [1732, 1730]
Costs per hop: [0.0, 1.0]
Total cost: 1.0
